# Published ParsBERT sentiment on every labeled review

`HooshvareLab/bert-fa-base-uncased-sentiment-digikala` is the only classifier in the two links. It is ParsBERT fine-tuned on the Open Data Mining Program Digikala set, 62,321 comments, with the same three names this project uses: `recommended`, `not_recommended`, and `no_idea`. On that small set's own test split the card reports accuracy 0.8268 and F1 0.8172. This project's labeled file is about 5.26 million reviews, and about 79% of them are `recommended`, against about 58% in the 62,321-comment set. The index order differs. This project uses `recommended=0`, `not_recommended=1`, `no_idea=2`. The published model uses `no_idea=0`, `not_recommended=1`, `recommended=2`. This notebook reads `id2label` from the model.

`EhsanShahbazi/digikala-comments` is not a model. It is a gated dump of 33,485,664 comments with no `recommendation_status`, so it has no class to score.

This notebook classifies every labeled review in this project's preprocessed CSV. Rows with an empty body or no `recommendation_status` are left out, because a metric needs a true class. Nothing is trained. A full pass on a T4 is several hours.


In [ ]:
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased-sentiment-digikala"
MAX_LENGTH = 128
BATCH_SIZE = 256
SEED = 2026

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
DRIVE_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
DATA_ZIP = "/content/data/digikala-comments-preprocessed.csv.zip"
LOCAL_CSV = "/content/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}


## Drive and this project's CSV

Mount this account's Drive for the score file. Download the same preprocessed CSV the fine-tune notebooks use. The notebook reads `body` and `recommendation_status` only.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import polars as pl
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

import gdown

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "runs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "logs").mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("This session has no GPU. Set the runtime to GPU, then run the notebook again.")
print("gpu:", torch.cuda.get_device_name(0))

zip_path = Path(DATA_ZIP)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=DRIVE_FILE_ID, output=str(zip_path), quiet=False)
if not zip_path.is_file():
    raise FileNotFoundError(f"Could not download Drive file {DRIVE_FILE_ID} to {zip_path}.")
with zipfile.ZipFile(zip_path) as archive:
    member = "digikala-comments-preprocessed.csv"
    names = [
        name
        for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX")
    ]
    if member not in names:
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        member = names[0]
    info = archive.getinfo(member)
    local_csv = Path(LOCAL_CSV)
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if (not local_csv.is_file()) or local_csv.stat().st_size != info.file_size:
        print("extracting", member, info.file_size, "bytes")
        with archive.open(info) as source, local_csv.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
print("csv:", local_csv, "bytes", local_csv.stat().st_size)


## Every labeled review

Keep rows with a non-empty `body` and a `recommendation_status` in the three classes. The score is on this whole labeled set, not on a 15% test cut.


In [ ]:
reviews = pl.read_csv(
    LOCAL_CSV,
    columns=["body", "recommendation_status"],
    schema_overrides={"body": pl.String, "recommendation_status": pl.String},
)
score_frame = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
)
print("labeled reviews", score_frame.height, "of", reviews.height)
print(score_frame.group_by("recommendation_status").len().sort("recommendation_status"))


## Load the published model and check four short sentences

The first print is the model's own index order. It is not this project's order.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model_id2label = {int(key): value for key, value in model.config.id2label.items()}
print("published id2label", model_id2label)
print("this project", LABEL_CODE)
for index, name in model_id2label.items():
    if name not in LABEL_CODE:
        raise KeyError(f"Published label {name!r} is not one of {LABELS}")

probes = [
    "پروژه به موقع تحویل شد و همه چیز خوب بود.",
    "سیب‌زمینی بی‌کیفیت بود.",
    "قیمت و کیفیت عالی",
    "خوب نبود اصلا",
]
device = torch.device("cuda")
model.to(device)
model.eval()
encoded = tokenizer(probes, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
encoded = {key: value.to(device) for key, value in encoded.items()}
with torch.no_grad():
    logits = model(**encoded).logits
predicted = logits.argmax(dim=-1).tolist()
for text, index in zip(probes, predicted):
    print(model_id2label[index], text)


## Score the whole labeled set

Precision, recall, and F1 below are macro averages over the three classes, the same reduction the fine-tune notebooks print. Per-class F1 is printed beside them. Twelve labeled reviews are printed with the true label and the published model's label.


In [ ]:
class ReviewSet(Dataset):
    def __init__(self, frame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average="macro", zero_division=0
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average=None, zero_division=0
    )
    report = {
        "accuracy": float(accuracy_score(actual, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    for index, name in enumerate(LABELS):
        report[f"{name}_precision"] = float(per_precision[index])
        report[f"{name}_recall"] = float(per_recall[index])
        report[f"{name}_f1"] = float(per_f1[index])
        report[f"{name}_support"] = int(per_support[index])
    return report


def our_ids(model_indexes):
    names = [model_id2label[int(index)] for index in model_indexes]
    return np.array([LABEL_CODE[name] for name in names], dtype=np.int64)


test_set = ReviewSet(score_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)
major, _minor = torch.cuda.get_device_capability()
shared = dict(
    output_dir="/tmp/hooshvare-sentiment-score",
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=major >= 7,
    report_to="none",
    dataloader_num_workers=2,
)
try:
    arguments = TrainingArguments(eval_strategy="no", **shared)
except TypeError:
    arguments = TrainingArguments(evaluation_strategy="no", **shared)

import inspect
token_kwargs = {}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    token_kwargs["processing_class"] = tokenizer
else:
    token_kwargs["tokenizer"] = tokenizer

holder = Trainer(
    model=model,
    args=arguments,
    data_collator=collator,
    **token_kwargs,
)
prediction = holder.predict(test_set)
actual = prediction.label_ids
predicted = our_ids(prediction.predictions.argmax(axis=-1))
report = metric_bundle(actual, predicted)
report.update({"split": "labeled", "model_name": MODEL_NAME, "rows": int(score_frame.height)})
print("labeled", {key: report[key] for key in ("accuracy", "precision", "recall", "f1")})
for name in LABELS:
    print(name, "f1", round(report[f"{name}_f1"], 4), "support", report[f"{name}_support"])

preview = score_frame.with_columns(
    pl.Series("predicted_id", predicted),
).sample(n=12, shuffle=True, seed=SEED)
for row in preview.iter_rows(named=True):
    body = (row["body"] or "").replace("\n", " ")
    if len(body) > 120:
        body = body[:117] + "..."
    print(row["recommendation_status"], "->", LABELS[int(row["predicted_id"])], "|", body)

slug = MODEL_NAME.replace("/", "--")
out_dir = Path(DRIVE_DIR) / "runs" / slug
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "test_metrics.json").write_text(json.dumps(report, indent=2))
log_path = Path(DRIVE_DIR) / "logs" / f"{slug}.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a") as handle:
    handle.write(json.dumps(report) + "\n")
print("wrote", out_dir / "test_metrics.json")
